In [0]:
USE CATALOG workspace;
USE SCHEMA ecom_poc;

In [0]:
CREATE OR REPLACE TEMP VIEW txns_parsed AS
WITH cleaned AS (
  SELECT
    b.*,   -- keep every original Bronze column (raw text plus audit columns)

    -- Text fields: trim, turn blanks / 'NULL' / 'NONE' into NULL, standardise case.
    CASE WHEN upper(trim(txn_id))         IN ('','NULL','NONE') THEN NULL ELSE trim(txn_id) END          AS txn_id_c,
    CASE WHEN upper(trim(order_id))       IN ('','NULL','NONE') THEN NULL ELSE trim(order_id) END        AS order_id_c,
    CASE WHEN upper(trim(payment_method)) IN ('','NULL','NONE') THEN NULL ELSE upper(trim(payment_method)) END AS payment_method_c,
    CASE WHEN upper(trim(currency))       IN ('','NULL','NONE') THEN NULL ELSE upper(trim(currency)) END AS currency_c,
    CASE WHEN upper(trim(txn_status))     IN ('','NULL','NONE') THEN NULL ELSE upper(trim(txn_status)) END AS txn_status_c,

    -- Safe type conversion: bad values become NULL instead of crashing.
    try_cast(trim(amount) AS DECIMAL(12,2))               AS amount_c,
    try_to_timestamp(trim(txn_ts), 'yyyy-MM-dd HH:mm:ss') AS txn_ts_c
  FROM bronze_transactions b
),
-- The set of order IDs that exist in the orders data (distinct, so the join below cannot multiply rows).
known_orders AS (
  SELECT DISTINCT trim(order_id) AS oid FROM bronze_orders
)
SELECT c.*,
  -- Same trick as orders: a list with one slot per rule, empty slots removed, glued with ';'.
  -- '' means the row is valid.
  array_join(filter(array(
    CASE WHEN c.txn_id_c IS NULL THEN 'MISSING_TXN_ID' END,

    -- Relationship check: missing order_id, or an order_id that matches no order (o.oid is NULL after the LEFT JOIN).
    CASE WHEN c.order_id_c IS NULL THEN 'MISSING_ORDER_ID'
         WHEN o.oid IS NULL THEN 'ORPHAN_ORDER_ID' END,

    CASE WHEN c.payment_method_c IS NULL THEN 'MISSING_PAYMENT_METHOD'
         WHEN c.payment_method_c NOT IN ('CARD','PAYPAL','BANK_TRANSFER','WALLET','COD') THEN 'INVALID_PAYMENT_METHOD' END,

    CASE WHEN c.amount_c IS NULL THEN 'AMOUNT_NOT_NUMERIC'
         WHEN c.amount_c < 0.01 OR c.amount_c > 10000000 THEN 'AMOUNT_OUT_OF_RANGE' END,

    CASE WHEN c.currency_c IS NULL THEN 'MISSING_CURRENCY'
         WHEN c.currency_c NOT IN ('USD','EUR','GBP','BDT') THEN 'UNSUPPORTED_CURRENCY' END,

    CASE WHEN c.txn_status_c IS NULL OR c.txn_status_c NOT IN ('SUCCESS','FAILED','PENDING','REFUNDED') THEN 'INVALID_TXN_STATUS' END,

    CASE WHEN c.txn_ts_c IS NULL THEN 'TXN_TS_INVALID'
         WHEN c.txn_ts_c > current_timestamp() THEN 'TXN_TS_FUTURE' END
  ), x -> x IS NOT NULL), ';') AS rejection_reason
FROM cleaned c
LEFT JOIN known_orders o ON c.order_id_c = o.oid;

In [0]:
-- Rows that broke at least one rule, saved with their ORIGINAL Bronze text values and the reasons.
CREATE OR REPLACE TABLE quarantine_transactions
COMMENT 'Transactions that failed at least one validation rule, with rejection reasons. Raw Bronze values kept.'
AS
SELECT txn_id, order_id, payment_method, amount, currency, txn_status, txn_ts,
       _ingested_at, _source_file, _batch_id, _row_hash, rejection_reason
FROM txns_parsed
WHERE rejection_reason <> '';

In [0]:
-- Among valid rows only, number the copies of each txn_id; lowest _row_hash gets rn = 1 (the winner).
CREATE OR REPLACE TEMP VIEW txns_ranked AS
SELECT *, ROW_NUMBER() OVER (PARTITION BY txn_id_c ORDER BY _row_hash ASC) AS rn
FROM txns_parsed
WHERE rejection_reason = '';

In [0]:
-- Winners: cleaned, correctly typed columns, one row per txn_id.
CREATE OR REPLACE TABLE silver_transactions
COMMENT 'Typed, validated transactions; exactly one row per txn_id'
AS
SELECT txn_id_c AS txn_id, order_id_c AS order_id, payment_method_c AS payment_method,
       amount_c AS amount, currency_c AS currency, txn_status_c AS txn_status, txn_ts_c AS txn_ts,
       _ingested_at, _source_file, _batch_id, _row_hash
FROM txns_ranked WHERE rn = 1;

-- Losers: valid rows beaten by the keep-one rule.
CREATE OR REPLACE TABLE duplicate_transactions
COMMENT 'Valid transactions that lost the keep-one rule (lowest _row_hash wins per txn_id)'
AS
SELECT txn_id_c AS txn_id, order_id_c AS order_id, payment_method_c AS payment_method,
       amount_c AS amount, currency_c AS currency, txn_status_c AS txn_status, txn_ts_c AS txn_ts,
       _ingested_at, _source_file, _batch_id, _row_hash, rn AS duplicate_rank
FROM txns_ranked WHERE rn > 1;

In [0]:
SELECT *, accepted + quarantined + duplicate_losers AS total,
       bronze_rows = accepted + quarantined + duplicate_losers AS reconciles
FROM (
  SELECT (SELECT COUNT(*) FROM bronze_transactions)     AS bronze_rows,
         (SELECT COUNT(*) FROM silver_transactions)     AS accepted,
         (SELECT COUNT(*) FROM quarantine_transactions) AS quarantined,
         (SELECT COUNT(*) FROM duplicate_transactions)  AS duplicate_losers
);

In [0]:
SELECT rejection_reason, COUNT(*) AS row_count
FROM quarantine_transactions
GROUP BY rejection_reason ORDER BY row_count DESC;

SELECT * FROM quarantine_transactions LIMIT 10;
SELECT * FROM duplicate_transactions LIMIT 10;